# Entrega 2 — Data Science: Governança, Qualidade e Análise Exploratória
### Violência contra a Mulher — BookSpace 

**Base:** SINAN — Violência Interpessoal/Autoprovocada (Ministério da Saúde), notificações de **2024**, recorte do grupo: **mulheres adultas com autor parceiro ou ex-parceiro**.

> ⚠️ **Aviso acadêmico** Este trabalho é um **protótipo acadêmico**. Nenhum número aqui substitui atendimento policial, psicológico ou de saúde. Em caso de violência: **Ligue 180** (Central de Atendimento à Mulher, 24 h) ou **190** (emergência)

## Passo 0 — Integrantes e configuração

In [ ]:
import re, unicodedata, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 170); pd.set_option("display.max_rows", 80)
plt.rcParams.update({"figure.facecolor": "white", "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "figure.dpi": 100, "font.size": 9})
COR, COR2, COR3 = "#00836C", "#E4572E", "#14284B"

GRUPO = "[PREENCHER: nome do grupo]"
INTEGRANTES = {   
    "Todos": "Governança e dicionário",
    "Guilherme Dorigon ": "Qualidade e tratamento dos dados",
    "Ana Julia Prado e Ana Clara Mantella": "EDA e Eng. Atributos",
    "Ana Julia Fernandes": "Modelagem e Avaliação",
}
print("Grupo:", GRUPO)
for nome, etapa in INTEGRANTES.items(): print(" -", nome, "->", etapa)

## 1. Governança, ética e LGPD (ficha da base)

| Pergunta | Resposta |
|---|---|
| **Nome da base** | SINAN — Violência Interpessoal/Autoprovocada (ficha de notificação individual; agravo CID-10 **Y09**, agressão). Arquivo usado: `bookspace_mulheres_parceiro_intimo_2024.csv` |
| **Órgão responsável** | Ministério da Saúde — Secretaria de Vigilância em Saúde (SINAN) / DATASUS |
| **Link da fonte** | Página do SINAN: <https://portalsinan.saude.gov.br/violencia-interpessoal-autoprovocada> · Dicionário de dados (SINAN NET v5.0, rev. 06/2015): <https://www.saude.sp.gov.br/resources/cve-centro-de-vigilancia-epidemiologica/tabnet/violencia/dic_dados_net_violencias_v5.pdf> |
| **Data de acesso** | 23/10/2026 |
| **Licença / termos de uso** |  |
| **Granularidade** | Cada linha é **uma notificação** (um atendimento notificado), não uma mulher. O arquivo não tem número de notificação nem nome; a mesma mulher pode aparecer mais de uma vez |
| **Período e abrangência** | Notificações de 01/01/2024 a 31/12/2024 (confirmado em `DT_NOTIFIC`), Brasil inteiro (todas as UFs) |
| **Recorte do grupo** | Só sexo feminino, idade ≥ 18 e autor parceiro ou ex-parceiro (cônjuge, ex-cônjuge, namorado ou ex-namorado). O diagnóstico abaixo **verifica** esse recorte|
| **Dado pessoal ou sensível?** | Sim, em estrutura: raça/cor, orientação sexual, identidade de gênero, deficiência/transtorno (inclusive um campo de **texto livre**, `DEF_ESPEC`), gravidez, tipo de violência. O arquivo **não traz** nome, CPF, cartão SUS, endereço nem telefone (as colunas existem na ficha oficial, mas não neste arquivo; ver seção 3) |
| **Risco de reidentificação** | Existe: município + data + ano de nascimento + tipo de violência pode apontar uma pessoa, sobretudo em cidade pequena. **Mitigação:** a base tratada não leva colunas de município, ano de nascimento nem texto livre; só UF/região, faixa etária e mês; cruzamentos com poucos casos (n < 10) não são publicados; ninguém tenta identificar pessoas (princípio da necessidade, LGPD art. 6º, III) |
| **Limitações** | **Subnotificação:** só aparece quem chegou a um serviço que notifica; mede o que foi **registrado**, não a incidência. **Viés de seleção** (quem procura saúde). Variação entre UFs no hábito de notificar. Muitos campos "Ignorado". Um mesmo caso pode ter mais de uma notificação. Sem informação do que aconteceu depois (ver seção 3.1: `EVOLUCAO` quase vazia) |
| **Por que serve ao app BookSpace?** | A ficha registra tipos de violência, meios de agressão, vínculo com o autor e **se a violência já ocorreu outras vezes**, o mesmo tipo de informação que o diário de ocorrências do app guarda. Isso permite analisar padrões reais e, na Entrega 3, testar um modelo sobre `OUT_VEZES` |


**Autor:** Todos

## 2. Carga da base e dicionário de dados

In [ ]:
ARQUIVO = "bookspace_mulheres_parceiro_intimo_2024.csv"
caminhos = [ARQUIVO, "dados/" + ARQUIVO, "../dados/" + ARQUIVO, "data-science/dados/" + ARQUIVO, "/mnt/user-data/uploads/" + ARQUIVO]
achado = next((p for p in caminhos if os.path.exists(p)), None)
if achado is None:
    from google.colab import files
    print("Arquivo não encontrado. Envie", ARQUIVO)
    up = files.upload(); achado = next(iter(up))

# dtype=str: pra preserva zeros a esquerda 
bruto = pd.read_csv(achado, dtype=str, low_memory=False)
print("Arquivo lido de:", achado)
print("Linhas x colunas:", bruto.shape)
display(bruto[["DT_NOTIFIC", "SG_UF_NOT", "IDADE_ANOS", "CS_SEXO", "CS_RACA", "OUT_VEZES", "VIOL_FISIC", "AG_AMEACA", "REL_CONJ"]].head(6))

A configuração das colunas. Os códigos seguem o dicionário oficial do SINAN

In [ ]:
ALVO = "OUT_VEZES"            # "A violência ocorreu outras vezes?" 1=Sim 2=Não 9=Ignorado 
COL_DATA = "DT_NOTIFIC"        # data principal 
COL_ID = None                  # o arquivo não tem número de notificação

# rótulos dos códigos (dicionário SINAN NET v5.0). 9/99 = Ignorado; 8/88 = Não se aplica
SN = {"1": "Sim", "2": "Não", "9": None}
ROTULOS = {
    "CS_RACA":    {"1": "Branca", "2": "Preta", "3": "Amarela", "4": "Parda", "5": "Indígena", "9": None},
    "CS_ESCOL_N": {"00": "Analfabeta", "01": "1ª-4ª série EF incompleta", "02": "4ª série EF completa", "03": "5ª-8ª série EF incompleta",
                   "04": "EF completo", "05": "EM incompleto", "06": "EM completo", "07": "Superior incompleto", "08": "Superior completo",
                   "09": None, "10": "Não se aplica"},
    "SIT_CONJUG": {"1": "Solteira", "2": "Casada/união consensual", "3": "Viúva", "4": "Separada", "8": "Não se aplica", "9": None},
    "CS_GESTANT": {"1": "Gestante (1º tri)", "2": "Gestante (2º tri)", "3": "Gestante (3º tri)", "4": "Gestante (IG ignorada)", "5": "Não gestante", "6": "Não se aplica", "9": None},
    "LOCAL_OCOR": {"01": "Residência", "02": "Habitação coletiva", "03": "Escola", "04": "Local de prática esportiva", "05": "Bar ou similar",
                   "06": "Via pública", "07": "Comércio/serviços", "08": "Indústria/construção", "09": "Outro", "99": None},
    "CICL_VID":   {"1": "Criança", "2": "Adolescente", "3": "Jovem", "4": "Adulto", "5": "Idoso", "9": None},   
    "AUTOR_SEXO": {"1": "Masculino", "2": "Feminino", "3": "Ambos", "9": None},
    "NUM_ENVOLV": {"1": "Um", "2": "Dois ou mais", "9": None},
    "VIOL_MOTIV": {"01": "Sexismo", "02": "LGBTfobia", "03": "Racismo", "04": "Intolerância religiosa", "05": "Xenofobia", "06": "Conflito geracional",
                   "07": "Situação de rua", "08": "Deficiência", "09": "Outros", "88": "Não se aplica", "99": None},
    "ORIENT_SEX": {"1": "Heterossexual", "2": "Homossexual", "3": "Bissexual", "8": "Não se aplica", "9": None},
}
COLS_SN = (["OUT_VEZES", "AUTOR_ALCO", "LES_AUTOP", "DEF_TRANS"]
           + ["VIOL_FISIC", "VIOL_PSICO", "VIOL_TORT", "VIOL_SEXU", "VIOL_TRAF", "VIOL_FINAN", "VIOL_NEGLI", "VIOL_INFAN", "VIOL_LEGAL", "VIOL_OUTR"]
           + ["AG_FORCA", "AG_ENFOR", "AG_OBJETO", "AG_CORTE", "AG_QUENTE", "AG_ENVEN", "AG_FOGO", "AG_AMEACA", "AG_OUTROS"]
           + ["REL_CONJ", "REL_EXCON", "REL_NAMO", "REL_EXNAM"]
           + ["REDE_SAU", "ASSIST_SOC", "ATEND_MULH", "DELEG_MULH", "MPU", "DEFEN_PUBL"])    
for c in COLS_SN: ROTULOS[c] = SN
COLS_CAT = list(ROTULOS.keys())
TIPOS_VIOL = {"VIOL_FISIC": "Física", "VIOL_PSICO": "Psicológica/moral", "VIOL_SEXU": "Sexual", "VIOL_FINAN": "Financeira/econômica", "VIOL_TORT": "Tortura"}
MEIOS_AGR = {"AG_FORCA": "Força corporal", "AG_AMEACA": "Ameaça", "AG_ENFOR": "Enforcamento", "AG_OBJETO": "Objeto contundente",
             "AG_CORTE": "Objeto perfurocortante", "AG_FOGO": "Arma de fogo", "AG_QUENTE": "Substância quente", "AG_ENVEN": "Envenenamento"}
ENCAMINH = ["REDE_SAU", "ASSIST_SOC", "ATEND_MULH", "DELEG_MULH", "MPU", "DEFEN_PUBL"]

COLS_NUM = ["IDADE_ANOS", "DIAS_ATE_NOTIF", "HORA"]          # DIAS_ATE_NOTIF e HORA são criar na seção 4
LIMITES = {"IDADE_ANOS": (18, 100), "DIAS_ATE_NOTIF": (0, 1825), "HORA": (0, 23)}   
# 1825 dias = 5 anos

UF = {"11":"RO","12":"AC","13":"AM","14":"RR","15":"PA","16":"AP","17":"TO","21":"MA","22":"PI","23":"CE","24":"RN","25":"PB","26":"PE","27":"AL","28":"SE",
      "29":"BA","31":"MG","32":"ES","33":"RJ","35":"SP","41":"PR","42":"SC","43":"RS","50":"MS","51":"MT","52":"GO","53":"DF"}
REGIAO = {**{k:"Norte" for k in ["11","12","13","14","15","16","17"]}, **{k:"Nordeste" for k in ["21","22","23","24","25","26","27","28","29"]},
          **{k:"Sudeste" for k in ["31","32","33","35"]}, **{k:"Sul" for k in ["41","42","43"]}, **{k:"Centro-Oeste" for k in ["50","51","52","53"]}}

assert set(COLS_CAT + [COL_DATA, "IDADE_ANOS", "HORA_OCOR", "DT_OCOR", "SG_UF_NOT"]) <= set(bruto.columns), "Há coluna da configuração que não existe na base."
print("Colunas categóricas analisadas:", len(COLS_CAT), "| numéricas:", COLS_NUM)

**Dicionário de dados — uma linha por coluna (160 colunas).** A coluna `fonte` diz de onde vem o significado: **oficial** = dicionário SINAN NET v5.0 lido pelo grupo; **a confirmar** = o campo não está nesse dicionário (versão mais nova da ficha ou campo interno do sistema); o significado vem do nome e precisa ser conferido no dicionário atual. `IDADE_ANOS` não existe no SINAN: foi derivada de `NU_IDADE_N` por quem preparou o arquivo.

In [ ]:
def info_col(c):
    """(significado, tipo estatístico, unidade, fonte) de cada coluna do arquivo."""
    o, a = "oficial", "a confirmar"
    fixos = {
     "TP_NOT": ("Tipo de notificação (2 = individual)", "qualitativa nominal", "—", o),
     "ID_AGRAVO": ("Agravo notificado, CID-10 (Y09 = agressão)", "qualitativa nominal", "—", o),
     "DT_NOTIFIC": ("Data de preenchimento da notificação", "data", "dd/mm/aaaa", o),
     "SEM_NOT": ("Semana epidemiológica da notificação (AAAASS)", "qualitativa ordinal", "semana", o),
     "NU_ANO": ("Ano da notificação", "quantitativa discreta", "ano", o),
     "SG_UF_NOT": ("UF da unidade notificadora (código IBGE)", "qualitativa nominal", "—", o),
     "ID_MUNICIP": ("Município da unidade notificadora (código IBGE, 6 dígitos)", "qualitativa nominal", "—", o),
     "DT_OCOR": ("Data da ocorrência da violência", "data", "dd/mm/aaaa", o),
     "SEM_PRI": ("Semana epidemiológica da ocorrência (AAAASS)", "qualitativa ordinal", "semana", o),
     "ANO_NASC": ("Ano de nascimento", "quantitativa discreta", "ano", a),
     "NU_IDADE_N": ("Idade codificada: 1º dígito = unidade (4 = anos) + valor (4033 = 33 anos)", "qualitativa nominal (código)", "—", o),
     "IDADE_ANOS": ("Idade em anos (derivada de NU_IDADE_N por quem preparou o arquivo)", "quantitativa discreta", "anos", a),
     "CS_SEXO": ("Sexo (F = feminino)", "qualitativa nominal", "—", o),
     "CS_GESTANT": ("Gestante / trimestre", "qualitativa nominal", "—", o),
     "CS_RACA": ("Raça/cor autodeclarada", "qualitativa nominal", "—", o),
     "CS_ESCOL_N": ("Escolaridade", "qualitativa ordinal", "—", o),
     "SG_UF": ("UF de residência (código IBGE)", "qualitativa nominal", "—", o),
     "ID_MN_RESI": ("Município de residência (código IBGE)", "qualitativa nominal", "—", o),
     "ID_PAIS": ("País de residência", "qualitativa nominal", "—", o),
     "NDUPLIC": ("Marca de duplicidade (rotina interna do sistema)", "qualitativa nominal", "—", o),
     "DT_INVEST": ("Data de investigação", "data", "dd/mm/aaaa", a),
     "ID_OCUPA_N": ("Ocupação (código CBO)", "qualitativa nominal", "—", o),
     "SIT_CONJUG": ("Situação conjugal", "qualitativa nominal", "—", o),
     "DEF_TRANS": ("Possui deficiência/transtorno?", "qualitativa nominal (binária)", "—", o),
     "DEF_ESPEC": ("Outras deficiências/síndromes — TEXTO LIVRE (dado sensível)", "texto livre", "—", o),
     "SG_UF_OCOR": ("UF de ocorrência (código IBGE)", "qualitativa nominal", "—", o),
     "ID_MN_OCOR": ("Município de ocorrência (código IBGE)", "qualitativa nominal", "—", o),
     "HORA_OCOR": ("Hora da ocorrência (hh:mm)", "quantitativa contínua", "hora", o),
     "LOCAL_OCOR": ("Local de ocorrência", "qualitativa nominal", "—", o),
     "LOCAL_ESPE": ("Local de ocorrência 'Outro' — texto livre", "texto livre", "—", o),
     "OUT_VEZES": ("A violência ocorreu outras vezes? (1 Sim, 2 Não, 9 Ignorado)", "qualitativa nominal (binária)", "—", o),
     "LES_AUTOP": ("A lesão foi autoprovocada?", "qualitativa nominal (binária)", "—", o),
     "VIOL_ESPEC": ("Outro tipo de violência — texto livre", "texto livre", "—", o),
     "AG_ESPEC": ("Outro meio de agressão — texto livre", "texto livre", "—", o),
     "SEX_ESPEC": ("Outro tipo de violência sexual — texto livre", "texto livre", "—", o),
     "NUM_ENVOLV": ("Número de envolvidos (1 ou 2 ou mais)", "qualitativa ordinal", "—", o),
     "AUTOR_SEXO": ("Sexo do provável autor", "qualitativa nominal", "—", o),
     "AUTOR_ALCO": ("Suspeita de uso de álcool pelo autor", "qualitativa nominal (binária)", "—", o),
     "REL_ESPEC": ("Outra relação com o autor — texto livre", "texto livre", "—", o),
     "REL_TRAB": ("Violência relacionada ao trabalho?", "qualitativa nominal (binária)", "—", o),
     "REL_CAT": ("Emitida comunicação de acidente de trabalho (CAT)?", "qualitativa nominal", "—", o),
     "CIRC_LESAO": ("Circunstância da lesão (CID-10, cap. XX)", "qualitativa nominal", "—", o),
     "CLASSI_FIN": ("Classificação final do caso", "qualitativa nominal", "—", a),
     "EVOLUCAO": ("Evolução do caso", "qualitativa nominal", "—", a),
     "DT_OBITO": ("Data do óbito", "data", "dd/mm/aaaa", a),
     "DT_DIGITA": ("Data de digitação no sistema", "data", "dd/mm/aaaa", a),
     "TPUNINOT": ("Tipo de unidade notificadora", "qualitativa nominal", "—", a),
     "ORIENT_SEX": ("Orientação sexual", "qualitativa nominal", "—", o),
     "IDENT_GEN": ("Identidade de gênero", "qualitativa nominal", "—", o),
     "VIOL_MOTIV": ("Violência motivada por (sexismo, racismo...)", "qualitativa nominal", "—", o),
     "CICL_VID": ("Ciclo de vida do provável autor", "qualitativa ordinal", "—", o),
     "DT_ENCERRA": ("Data de encerramento do caso", "data", "dd/mm/aaaa", o),
    }
    if c in fixos: return fixos[c]
    pref = [("VIOL_", "Tipo de violência ocorrida", "qualitativa nominal (binária)", o), ("AG_", "Meio de agressão", "qualitativa nominal (binária)", o),
            ("SEX_", "Tipo de violência sexual", "qualitativa nominal (binária)", o), ("PROC_", "Procedimento realizado (violência sexual)", "qualitativa nominal (binária)", o),
            ("REL_", "Relação do provável autor com a vítima", "qualitativa nominal (binária)", o), ("DEF_", "Tipo de deficiência", "qualitativa nominal (binária)", o),
            ("TRAN_", "Tipo de transtorno", "qualitativa nominal (binária)", o), ("PEN_", "Tipo de penetração (ficha mais nova)", "qualitativa nominal", a),
            ("CONS_", "Consequência da violência (ficha mais antiga)", "qualitativa nominal", a), ("LESAO_", "Lesão: natureza/parte do corpo (ficha mais antiga)", "qualitativa nominal", a),
            ("ENC_", "Encaminhamento (ficha mais antiga)", "qualitativa nominal", a), ("DT_TRANS", "Data de transferência do registro entre esferas do sistema (campo interno)", "data", a),
            ("REDE_", "Encaminhamento para a rede", "qualitativa nominal (binária)", o), ("ASSIST_", "Encaminhamento: assistência social", "qualitativa nominal (binária)", o),
            ("ATEND_", "Encaminhamento: rede de atendimento à mulher", "qualitativa nominal (binária)", o), ("CONS_TUTEL", "Encaminhamento: conselho tutelar", "qualitativa nominal (binária)", o),
            ("DELEG", "Encaminhamento: delegacia", "qualitativa nominal (binária)", o), ("DIR_", "Encaminhamento: centro de direitos humanos", "qualitativa nominal (binária)", o),
            ("MPU", "Encaminhamento: Ministério Público", "qualitativa nominal (binária)", o), ("INFAN_", "Encaminhamento: justiça da infância e juventude", "qualitativa nominal (binária)", o),
            ("DEFEN_", "Encaminhamento: defensoria pública", "qualitativa nominal (binária)", o)]
    for p, s, t, f in pref:
        if c.startswith(p): return (f"{s} ({c})", t, "—", f)
    return ("Campo sem descrição neste dicionário", "qualitativa nominal", "—", a)

dicionario = pd.DataFrame([(c, *info_col(c)) for c in bruto.columns], columns=["coluna", "significado", "tipo estatístico", "unidade", "fonte"])
dicionario["% vazio"] = (bruto.isna().mean() * 100).round(1).values
print("Colunas descritas:", len(dicionario), "| fonte oficial:", int((dicionario.fonte == "oficial").sum()), "| a confirmar:", int((dicionario.fonte == "a confirmar").sum()))
display(dicionario.head(60))
os.makedirs("dados", exist_ok=True)
dicionario.to_csv("dados/dicionario_de_dados.csv", index=False)

**Autor:** Guilherme Dorigon

## 3. Diagnóstico de qualidade
Medir antes de corrigir: **completude, validade, consistência, unicidade/integridade e acurácia**.

### 3.1 Completude: quantos valores faltam em cada coluna?

In [ ]:
n = len(bruto)
pct_exato = bruto.isna().mean() * 100                       # sem arredondar: 100% = nenhuma linha preenchida
pct_nulos = pct_exato.round(1).sort_values(ascending=False)
vazias = pct_exato[pct_exato == 100].index.tolist()
quase = pct_exato[(pct_exato >= 90) & (pct_exato < 100)].round(2).sort_values(ascending=False)
print(f"{len(vazias)} colunas 100% vazias:", vazias)
print(f"{len(quase)} colunas com 90% a 99,99% vazias (quase sem informação). Linhas preenchidas nas 8 mais vazias:", {c: int(bruto[c].notna().sum()) for c in quase.head(8).index})
print("\nCampos de desfecho (o que aconteceu DEPOIS):")
print("  EVOLUCAO preenchida em", int(bruto['EVOLUCAO'].notna().sum()), "de", n, "linhas")
print("  CLASSI_FIN preenchida em", int(bruto['CLASSI_FIN'].notna().sum()), "de", n, "linhas")
print("  DT_OBITO preenchida em", int(bruto['DT_OBITO'].notna().sum()), "linhas")

analise = pct_exato[pct_exato < 90].sort_values(ascending=False).head(25)
ax = analise.sort_values().plot.barh(color=COR, figsize=(7, 6))
ax.set_xlabel("% de valores vazios (NaN)"); ax.set_ylabel("coluna"); ax.set_title("Completude: 25 colunas com mais vazios entre as que têm algum dado (< 90% vazio)")
plt.tight_layout(); plt.show()

### 3.2 Buracos disfarçados
`Ignorado` (código 9/99), `Não se aplica` mal usado e códigos de ocupação genéricos **não são informação**. A tabela conta os códigos de *ignorado* nas colunas analisadas.

In [ ]:
IGN = {c: {k for k, v in ROTULOS[c].items() if v is None} for c in COLS_CAT}
disf = pd.DataFrame({
    "ignorados (código 9/99)": {c: int(bruto[c].isin(IGN[c]).sum()) for c in COLS_CAT},
    "vazios (NaN)": {c: int(bruto[c].isna().sum()) for c in COLS_CAT}})
disf["% ignorado"] = (disf.iloc[:, 0] / n * 100).round(1)
disf["% ignorado+vazio"] = ((disf.iloc[:, 0] + disf.iloc[:, 1]) / n * 100).round(1)
display(disf[disf["ignorados (código 9/99)"] > 0].sort_values("% ignorado", ascending=False).head(20))

# ocupação: códigos genéricos e texto 'XXX'
oc = bruto["ID_OCUPA_N"].dropna()
ocup_99 = int(oc.str.startswith("99").sum()); ocup_xxx = int((oc == "XXX").sum())
print(f"\nID_OCUPA_N: {len(oc)} preenchidos; {ocup_99} ({ocup_99/len(oc)*100:.1f}%) começam com '99' (ex.: {oc[oc.str.startswith('99')].value_counts().head(3).to_dict()}) e {ocup_xxx} são 'XXX'")
print("Exemplo de buraco disfarçado em hora:", bruto.loc[~bruto["HORA_OCOR"].fillna("00:00").str.match(r"^([01]\d|2[0-3]):[0-5]\d$"), "HORA_OCOR"].tolist())

for c in ["OUT_VEZES", "CS_RACA", "CS_ESCOL_N", "SIT_CONJUG", "AUTOR_ALCO"]:
    print(f"{c}: ", bruto[c].value_counts(dropna=False).to_dict())

### 3.3 Validade: valores fora do domínio ou da faixa plausível

In [ ]:
# (a) códigos fora do domínio do dicionário
fora_dom = {c: int((bruto[c].notna() & ~bruto[c].isin(ROTULOS[c].keys())).sum()) for c in COLS_CAT}
print("Códigos fora do domínio (por coluna):", {k: v for k, v in fora_dom.items() if v} or "nenhum")

# (b) numéricas e datas
bruto_idade = pd.to_numeric(bruto["IDADE_ANOS"], errors="coerce")
dt_not = pd.to_datetime(bruto["DT_NOTIFIC"], errors="coerce"); dt_oc = pd.to_datetime(bruto["DT_OCOR"], errors="coerce")
bruto_dias = (dt_not - dt_oc).dt.days
hora_ok = bruto["HORA_OCOR"].dropna().str.match(r"^([01]\d|2[0-3]):[0-5]\d$")
val = pd.DataFrame({
    "valores inválidos": [int((bruto_idade.notna() & ~bruto_idade.between(*LIMITES["IDADE_ANOS"])).sum()),
                          int((bruto_dias.notna() & ~bruto_dias.between(*LIMITES["DIAS_ATE_NOTIF"])).sum()),
                          int((~hora_ok).sum()),
                          int((bruto["DT_NOTIFIC"].notna() & dt_not.isna()).sum()), int((bruto["DT_OCOR"].notna() & dt_oc.isna()).sum())],
    "faixa válida": ["18 a 100 anos", "0 a 1825 dias (ocorrência até 5 anos antes)", "00:00 a 23:59", "data real", "data real"]},
    index=["IDADE_ANOS", "dias entre ocorrência e notificação", "HORA_OCOR (formato)", "DT_NOTIFIC ilegível", "DT_OCOR ilegível"])
display(val)
print("\nExemplos reais:")
print("Idades inválidas:", sorted(bruto_idade[~bruto_idade.between(18, 100)].tolist()))
print("Ocorrência mais antiga:", dt_oc.min().date(), "→ notificada em", dt_not[dt_oc.idxmin()].date(), f"({int(bruto_dias[dt_oc.idxmin()])} dias, cerca de {bruto_dias[dt_oc.idxmin()]/365:.0f} anos)")
print("Ocorrências com mais de 5 anos antes da notificação:", int((bruto_dias > 1825).sum()))

### 3.4 Consistência: regras que o próprio dicionário impõe e grafias diferentes

In [ ]:
ano = pd.to_numeric(bruto["ANO_NASC"], errors="coerce")
dif_idade = ((2024 - ano) - bruto_idade).abs()
sex_nao = bruto["VIOL_SEXU"].isin(["2", "9"]) & bruto["SEX_ESTUPR"].notna() & (bruto["SEX_ESTUPR"] != "8")
sex_sim = (bruto["VIOL_SEXU"] == "1") & (bruto["SEX_ESTUPR"] == "8")
dt_enc = pd.to_datetime(bruto["DT_ENCERRA"], errors="coerce"); dt_dig = pd.to_datetime(bruto["DT_DIGITA"], errors="coerce")
vinc = bruto[["REL_CONJ", "REL_EXCON", "REL_NAMO", "REL_EXNAM"]].eq("1")
cons = pd.DataFrame([
    ("Idade × ano de nascimento (2024 − ANO_NASC difere de IDADE_ANOS em mais de 1 ano)", int((dif_idade > 1).sum())),
    ("Violência sexual = Não/Ign., mas subtipo sexual preenchido (regra: deveria ser 'Não se aplica')", int(sex_nao.sum())),
    ("Violência sexual = Sim, mas estupro = 'Não se aplica'", int(sex_sim.sum())),
    ("Data de encerramento anterior à data de notificação", int((dt_enc < dt_not).sum())),
    ("Data de digitação anterior à data de notificação", int((dt_dig < dt_not).sum())),
    ("Data de ocorrência posterior à notificação (regra do dicionário: ocorrência ≤ notificação)", int((dt_oc > dt_not).sum())),
    ("Notificação sem nenhum vínculo parceiro/ex (recorte do grupo)", int((~vinc.any(axis=1)).sum())),
    ("Notificação com 2 ou mais vínculos marcados ao mesmo tempo", int((vinc.sum(axis=1) > 1).sum())),
    ("Sexo diferente de feminino / idade < 18 (recorte do grupo)", int(((bruto["CS_SEXO"] != "F") | (bruto_idade < 18)).sum())),
    ("UF de residência diferente da UF de notificação (informativo, não é erro)", int((bruto["SG_UF"] != bruto["SG_UF_NOT"]).sum())),
], columns=["regra verificada", "linhas que violam"])
display(cons)

# grafias: texto livre de deficiência/transtorno
STOP = {"E", "DE", "DA", "DO", "COM", "TRANSTORNO", "USUARIA"}
def chave_texto(s):
    """Chave comparável: sem acento, maiúsculas, sem pontuação, sem palavras de ligação e com as palavras em ordem alfabética."""
    t = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().upper()
    t = re.sub(r"[^A-Z0-9 ]", " ", t)
    return " ".join(sorted(w for w in t.split() if w not in STOP))
txt = bruto["DEF_ESPEC"].dropna().str.strip()
print("\nDEF_ESPEC (texto livre):", txt.nunique(), "grafias brutas →", txt.map(chave_texto).nunique(), "depois de padronizar (acento, caixa, palavras de ligação e ordem)")
exemplo = txt[txt.map(chave_texto) == "ANSIEDADE DEPRESSAO"].str.upper().value_counts().head(5)
print("Exemplo: estas grafias diferentes viram a mesma chave 'ANSIEDADE DEPRESSAO':", exemplo.to_dict())

### 3.5 Unicidade e integridade

In [ ]:
dup_exatas = int(bruto.duplicated().sum())
chave_quase = ["DT_NOTIFIC", "ID_MUNICIP", "DT_OCOR", "ANO_NASC", "CS_RACA", "CS_ESCOL_N", "SIT_CONJUG", "VIOL_FISIC", "VIOL_PSICO", "VIOL_SEXU", "OUT_VEZES", "AG_FORCA", "AG_AMEACA"]
dup_quase = int(bruto.duplicated(chave_quase).sum())
print(f"Linhas totalmente idênticas (a partir da 2ª cópia): {dup_exatas}")
print(f"Linhas idênticas em 13 campos-chave (possível duplicata ou casos parecidos): {dup_quase}")
print("Não há número de notificação (COL_ID = None): a unicidade só pode ser avaliada por linhas idênticas e campos-chave.")
display(bruto[bruto.duplicated(keep=False)].sort_values(["DT_NOTIFIC", "ID_MUNICIP"])[["DT_NOTIFIC", "ID_MUNICIP", "ANO_NASC", "CS_RACA", "OUT_VEZES", "DT_DIGITA"]])

# integridade referencial: município pertence à UF informada? (os 2 primeiros dígitos do código IBGE do município = código da UF)
integ_not = int((bruto["ID_MUNICIP"].str[:2] != bruto["SG_UF_NOT"]).sum())
integ_res = int((bruto["ID_MN_RESI"].str[:2] != bruto["SG_UF"]).sum())
oc_ok = bruto["ID_MN_OCOR"].notna() & bruto["SG_UF_OCOR"].notna()
integ_oco = int((bruto.loc[oc_ok, "ID_MN_OCOR"].str[:2] != bruto.loc[oc_ok, "SG_UF_OCOR"]).sum())
print(f"\nIntegridade município × UF: notificação {integ_not} | residência {integ_res} | ocorrência {integ_oco} linhas incoerentes")
print("Códigos de UF fora da tabela IBGE:", int((~bruto["SG_UF_NOT"].isin(UF.keys())).sum()))

### 3.6 Acurácia
Acurácia pergunta: **o dado reflete a realidade?** Isso não se descobre olhando só o arquivo; é preciso **comparar com outra fonte**. O que dá para fazer:

1. **Comparar o total** do arquivo com a mesma consulta no TabNet/DATASUS (violência interpessoal/autoprovocada, 2024, sexo feminino, ≥ 18 anos, autor parceiro/ex). Se os totais forem próximos, o arquivo está fiel à fonte. Preencha `TOTAL_TABNET` abaixo quando o grupo fizer a consulta.
2. **Plausibilidade interna** (feita na 3.4): idade × ano de nascimento e datas.
3. A **acurácia do que a vítima/profissional informou** (por exemplo, se "ocorreu outras vezes" é verdade) **não é mensurável** com dados públicos: vira limitação.

In [ ]:
TOTAL_TABNET = None   # [PREENCHER] total obtido no TabNet com os mesmos filtros; deixe None se ainda não consultou
if TOTAL_TABNET:
    print(f"Arquivo: {n} linhas | TabNet: {TOTAL_TABNET} | diferença: {n - TOTAL_TABNET} ({(n - TOTAL_TABNET) / TOTAL_TABNET * 100:+.2f}%)")
else:
    print(f"Arquivo: {n} linhas. Falta o total do TabNet para comparar (a consulta é um passo do grupo).")
ref = bruto["SG_UF_NOT"].map(UF).value_counts(normalize=True).mul(100).round(1)
print("Maior parte das notificações por UF (%):", ref.head(5).to_dict())

### 3.7 Resumo do diagnóstico

In [ ]:
ign_total = int(disf["ignorados (código 9/99)"].sum())
resumo = pd.DataFrame([
    ("Completude", f"{len(vazias)} colunas 100% vazias; HORA_OCOR {pct_nulos['HORA_OCOR']}% vazia; EVOLUCAO e CLASSI_FIN quase vazias", int(bruto.isna().sum().sum()), "descartar colunas vazias; não imputar hora; não usar desfecho"),
    ("Completude (disfarçada)", f"Código 9/99 (Ignorado) nas colunas analisadas; ocupação genérica '99xxxx'/'XXX'", ign_total + ocup_99 + ocup_xxx, "9/99 → NaN → 'Não informado'; ocupação fora da análise"),
    ("Validade", "Idade > 100; ocorrência > 5 anos antes da notificação; hora '00000'", int(val["valores inválidos"].iloc[:3].sum()), "transformar em NaN (e sinalizar)"),
    ("Consistência", "Idade × ano de nascimento; subtipo sexual com 'Não'; encerramento antes da notificação", int(cons["linhas que violam"].iloc[[0, 1, 2, 3, 4, 5]].sum()), "documentar; corrigir só o que não depende de adivinhar"),
    ("Unicidade", "Linhas totalmente idênticas (não há número de notificação)", dup_exatas, "remover as idênticas; manter as 'quase' iguais (podem ser casos distintos)"),
    ("Integridade", "Município × UF", integ_not + integ_res + integ_oco, "ok se zero"),
    ("Acurácia", "Comparar total com o TabNet; veracidade das respostas não mensurável", 0, "documentar como limitação"),
], columns=["dimensão", "problema encontrado", "ocorrências", "decisão"])
display(resumo)

**Autor:** Guilherme Dorigon

## 4. Tratamento dos dados
**A ordem importa:** padronizar textos e datas → ausentes disfarçados e inválidos → duplicatas → imputação (com critério) → sinalizar outliers → (só então) escalonar. Trabalhamos numa **cópia** (`df`); a base bruta (`bruto`) fica intocada e o arquivo original não é alterado.

### 4.1 Padronizar textos e datas (consistência)

In [ ]:
df = bruto.copy()
# textos: tirar espaços e padronizar caixa/acento (só o que é texto livre)
def sem_acento(s): return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
for c in df.columns:
    df[c] = df[c].astype("string").str.strip()
def padroniza_texto(s): return s.map(lambda v: sem_acento(v).upper().strip() if pd.notna(v) else v)
antes_txt = df["DEF_ESPEC"].nunique()
df["DEF_ESPEC"] = df["DEF_ESPEC"].map(lambda v: chave_texto(v) if pd.notna(v) else v)      # chave_texto foi definida na seção 3.4
print(f"DEF_ESPEC (texto livre): {antes_txt} grafias → {df['DEF_ESPEC'].nunique()} depois de padronizar (a coluna sai da base tratada por LGPD, mas o exemplo vale para qualquer texto livre)")

# datas: texto → datetime (data impossível viraria NaT)
for c in ["DT_NOTIFIC", "DT_OCOR", "DT_DIGITA", "DT_ENCERRA"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")
print("Datas ilegíveis após conversão (NaT) em DT_NOTIFIC/DT_OCOR:", int(df["DT_NOTIFIC"].isna().sum()), "/", int(df["DT_OCOR"].isna().sum()))
print("Período de notificação:", df["DT_NOTIFIC"].min().date(), "a", df["DT_NOTIFIC"].max().date())

# numéricas
df["IDADE_ANOS"] = pd.to_numeric(df["IDADE_ANOS"], errors="coerce")
df["DIAS_ATE_NOTIF"] = (df["DT_NOTIFIC"] - df["DT_OCOR"]).dt.days
h = df["HORA_OCOR"].where(df["HORA_OCOR"].str.match(r"^([01]\d|2[0-3]):[0-5]\d$", na=False))
df["HORA"] = pd.to_numeric(h.str[:2], errors="coerce")        # hora cheia (0 a 23)
df["HORA_INVALIDA"] = df["HORA_OCOR"].notna() & df["HORA"].isna()
print("Exemplos: DT_OCOR 1968 →", int(df["DIAS_ATE_NOTIF"].max()), "dias; hora '03:00' → 3")

### 4.2 Ausentes disfarçados e valores inválidos → NaN
"Ignorado" (9/99) não é uma categoria real: vira ausente. Valores fora da faixa em `LIMITES` viram NaN, mas **ficam sinalizados** (coluna-bandeira) para o leitor saber quantos eram.

In [ ]:
n_ign = {}
for c in COLS_CAT:
    marca = df[c].isin(IGN[c])
    n_ign[c] = int(marca.sum())
    df[c] = df[c].map(ROTULOS[c])                 # decodifica; Ignorado (None) e código desconhecido viram NaN

# inválidos numéricos → NaN + bandeira
df["IDADE_INVALIDA"] = df["IDADE_ANOS"].notna() & ~df["IDADE_ANOS"].between(*LIMITES["IDADE_ANOS"])
df["OCORRENCIA_ANTIGA"] = df["DIAS_ATE_NOTIF"].notna() & ~df["DIAS_ATE_NOTIF"].between(*LIMITES["DIAS_ATE_NOTIF"])
df.loc[df["IDADE_INVALIDA"], "IDADE_ANOS"] = np.nan
df.loc[df["OCORRENCIA_ANTIGA"], "DIAS_ATE_NOTIF"] = np.nan

print("Ignorados convertidos em NaN (10 maiores):", dict(sorted({k: v for k, v in n_ign.items() if v}.items(), key=lambda x: -x[1])[:10]))
print("Idades inválidas → NaN:", int(df["IDADE_INVALIDA"].sum()), "| ocorrências > 5 anos → dias NaN:", int(df["OCORRENCIA_ANTIGA"].sum()), "| horas inválidas:", int(df["HORA_INVALIDA"].sum()))

### 4.3 Duplicatas
Só as linhas **totalmente idênticas** são removidas. Linhas parecidas em poucos campos (3.5) **não** se removem: sem número de notificação, podem ser duas mulheres diferentes.

In [ ]:
dup_mask = bruto.duplicated(keep="first")
n0 = len(df)
df = df.loc[~dup_mask.values].reset_index(drop=True)
print(f"{n0} → {len(df)} linhas ({n0 - len(df)} duplicatas removidas)")

### 4.4 Variáveis derivadas (sem usar informação futura)

In [ ]:
vinc_cols = {"REL_CONJ": "Cônjuge", "REL_EXCON": "Ex-cônjuge", "REL_NAMO": "Namorado(a)", "REL_EXNAM": "Ex-namorado(a)"}
sim = pd.DataFrame({k: df[k].eq("Sim") for k in vinc_cols})
def vinc(row):
    ativos = [vinc_cols[k] for k in vinc_cols if row[k]]
    return "Não informado" if not ativos else (ativos[0] if len(ativos) == 1 else "Mais de um vínculo")
df["VINCULO"] = sim.apply(vinc, axis=1)
df["N_TIPOS"] = df[[c for c in COLS_CAT if c.startswith("VIOL_") and c != "VIOL_MOTIV"]].eq("Sim").sum(axis=1)
df["N_MEIOS"] = df[[c for c in COLS_CAT if c.startswith("AG_")]].eq("Sim").sum(axis=1)
df["FAIXA_ETARIA"] = pd.cut(df["IDADE_ANOS"], [17, 24, 34, 44, 59, 200], labels=["18-24", "25-34", "35-44", "45-59", "60+"]).astype("string")
df["UF"] = df["SG_UF_NOT"].map(UF); df["REGIAO"] = df["SG_UF_NOT"].map(REGIAO)
df["MES_NOTIF"] = df["DT_NOTIFIC"].dt.month
df["DIA_SEMANA"] = df["DT_NOTIFIC"].dt.dayofweek.map({0: "seg", 1: "ter", 2: "qua", 3: "qui", 4: "sex", 5: "sáb", 6: "dom"})
df["ALVO_BIN"] = df[ALVO].map({"Sim": 1, "Não": 0})                  # 1 = ocorreu outras vezes; NaN = ignorado/vazio (fora do modelo)
print(df["VINCULO"].value_counts().to_dict())
print("Tipos de violência por notificação:", df["N_TIPOS"].value_counts().sort_index().to_dict())

### 4.5 Imputação (com critério)
Regra do curso: **numérica** → mediana (se assimétrica) + coluna-bandeira; **categórica** → categoria explícita "Não informado" (sem inventar a moda); **alvo** → nunca imputar. **Critério extra do grupo:** não imputar coluna com mais de 30% de ausentes, porque o valor preenchido passaria a ser, em sua maior parte, inventado.

In [ ]:
LIMITE_IMPUTACAO = 30
faltando = df[COLS_NUM].isna().mean().mul(100).round(1)
dist = pd.DataFrame({"média": df[COLS_NUM].mean(), "mediana": df[COLS_NUM].median(),
                     "assimetria": df[COLS_NUM].apply(lambda s: stats.skew(s.dropna())), "% faltando": faltando}).round(2)
dist["decisão"] = ["imputar mediana + bandeira (faltam poucos)" if f <= LIMITE_IMPUTACAO and c == "IDADE_ANOS"
                   else ("não imputar: faltante são casos de ocorrência antiga, outra população" if c == "DIAS_ATE_NOTIF"
                         else "não imputar: " + str(f) + "% faltando") for c, f in faltando.items()]
display(dist)

df["IDADE_IMPUTADA"] = df["IDADE_ANOS"].isna()
df["IDADE_ANOS"] = df["IDADE_ANOS"].fillna(df["IDADE_ANOS"].median())
df["FAIXA_ETARIA"] = pd.cut(df["IDADE_ANOS"], [17, 24, 34, 44, 59, 200], labels=["18-24", "25-34", "35-44", "45-59", "60+"]).astype("string")

for c in COLS_CAT:
    if c != ALVO:
        df[c] = df[c].fillna("Não informado")
df["FAIXA_ETARIA"] = df["FAIXA_ETARIA"].fillna("Não informado")
print("Idades imputadas:", int(df["IDADE_IMPUTADA"].sum()))
print("Alvo SEM imputar:", df[ALVO].value_counts(dropna=False).to_dict())
print("NaN restantes (esperado: alvo, dias, hora, UF/região se código desconhecido):", df[["ALVO_BIN", "DIAS_ATE_NOTIF", "HORA", "UF", "REGIAO"]].isna().sum().to_dict())

### 4.6 Outliers: sinalizar, não apagar
Valores **válidos** mas extremos não são erro. Sinalizamos pelo IQR e decidimos na modelagem (manter, `RobustScaler` ou log).

In [ ]:
out = {}
for c, bandeira in [("IDADE_ANOS", "IDADE_OUTLIER"), ("DIAS_ATE_NOTIF", "DIAS_OUTLIER")]:
    s = df.loc[~df["IDADE_IMPUTADA"], c] if c == "IDADE_ANOS" else df[c]
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    inf, sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df[bandeira] = df[c].notna() & ((df[c] < inf) | (df[c] > sup)) & ~(df["IDADE_IMPUTADA"] if c == "IDADE_ANOS" else False)
    out[c] = (int(df[bandeira].sum()), round(q1, 1), round(q3, 1), round(inf, 1), round(sup, 1))
display(pd.DataFrame(out, index=["qtd outliers (IQR)", "Q1", "Q3", "limite inferior", "limite superior"]).T)

### 4.7 Escalonamento (demonstração para a Entrega 3)
Só demonstração, em **cópia**. Lições: escalonar **não muda o formato** (a assimetria é a mesma) e, na modelagem, o scaler aprende (`fit`) **só no treino**.

In [ ]:
demo = df[["IDADE_ANOS", "DIAS_ATE_NOTIF"]].dropna().copy()
mm = pd.DataFrame(MinMaxScaler().fit_transform(demo), columns=demo.columns)
zs = pd.DataFrame(StandardScaler().fit_transform(demo), columns=demo.columns)
rb = pd.DataFrame(RobustScaler().fit_transform(demo), columns=demo.columns)
cmp_ = pd.DataFrame({"assimetria original": demo.apply(stats.skew), "min-max": mm.apply(stats.skew), "z-score": zs.apply(stats.skew), "robusto": rb.apply(stats.skew)}).round(3)
display(cmp_)
print("Min-max de DIAS_ATE_NOTIF: mediana =", round(mm["DIAS_ATE_NOTIF"].median(), 4), "(quase tudo espremido perto de 0 pelos extremos)")
fig, ax = plt.subplots(1, 3, figsize=(11, 3))
for a, (t, d) in zip(ax, [("dias (original)", demo["DIAS_ATE_NOTIF"]), ("dias (min-max)", mm["DIAS_ATE_NOTIF"]), ("dias (robusto)", rb["DIAS_ATE_NOTIF"])]):
    a.hist(d.clip(upper=d.quantile(.99)), bins=30, color=COR); a.set_title(t); a.set_xlabel("valor (cortado no percentil 99)"); a.set_ylabel("notificações")
plt.suptitle("Escalonar muda a escala, não o formato da distribuição", y=1.02); plt.tight_layout(); plt.show()

### 4.8 Tabela de decisões, antes × depois e arquivo tratado

In [ ]:
n_vazias = len(vazias)
decisoes = pd.DataFrame([
    ("Colunas 100% vazias", f"Não levar {n_vazias} colunas para a base tratada", "Sem informação nenhuma; só ocupam espaço", f"160 → colunas da base tratada: ver abaixo"),
    ("LGPD / minimização", "Não levar município, ano de nascimento, datas exatas de ocorrência/digitação/encerramento, textos livres nem ocupação", "Reduz risco de reidentificação; ocupação ainda tinha 52% de códigos genéricos", "Ficam UF, região, faixa etária, mês e dias até notificar"),
    ("Textos e datas", "Strip, caixa/acento e datas → datetime", "Mesmo conceito escrito de formas diferentes; data como texto não ordena", f"DEF_ESPEC: {antes_txt} → {df['DEF_ESPEC'].nunique()} grafias"),
    ("'Ignorado' (9/99)", "Código → NaN → 'Não informado' (menos no alvo)", "Não é informação; manter a contagem visível", f"{sum(n_ign.values())} ignorados nas colunas analisadas"),
    ("Valores inválidos", "Idade fora de 18-100, dias fora de 0-1825, hora fora do formato → NaN + bandeira", "Idade 123 ou ocorrência de 1968 não são medidas confiáveis do caso", f"{int(df['IDADE_INVALIDA'].sum())} idades; {int(df['OCORRENCIA_ANTIGA'].sum())} ocorrências antigas; {int(df['HORA_INVALIDA'].sum())} hora"),
    ("Duplicatas", "Remover só linhas totalmente idênticas", "Contavam o mesmo registro duas vezes; as 'quase iguais' podem ser pessoas diferentes", f"{n0 - len(df)} linhas removidas"),
    ("Imputação numérica", "Mediana + bandeira só na idade; dias e hora NÃO", "Dias faltantes são ocorrências antigas; hora tem muito vazio (> 30%)", f"{int(df['IDADE_IMPUTADA'].sum())} idades imputadas"),
    ("Imputação categórica", "'Não informado' (sem usar a moda)", "Evita inventar uma categoria dominante", "ver tabela de frequências"),
    ("Alvo (OUT_VEZES)", "Não imputar; ignorado/vazio ficam fora do treino", "Inventar o alvo contaminaria o modelo", f"{int(df['ALVO_BIN'].isna().sum())} linhas sem alvo"),
    ("Outliers", "Manter e sinalizar (IQR)", "São valores válidos; tratar na modelagem", f"idade: {int(df['IDADE_OUTLIER'].sum())}; dias: {int(df['DIAS_OUTLIER'].sum())}"),
    ("Escalonamento", "Só demonstrar; aplicar na Entrega 3 após a divisão treino/teste", "Evitar vazamento de dados", "—"),
], columns=["tema", "o que fiz", "por quê", "impacto"])
display(decisoes)

COLS_FINAL = (["DT_NOTIFIC", "MES_NOTIF", "DIA_SEMANA", "UF", "REGIAO", "IDADE_ANOS", "FAIXA_ETARIA", "IDADE_IMPUTADA", "IDADE_INVALIDA", "IDADE_OUTLIER",
               "DIAS_ATE_NOTIF", "OCORRENCIA_ANTIGA", "DIAS_OUTLIER", "HORA", "HORA_INVALIDA"]
              + COLS_CAT + ["VINCULO", "N_TIPOS", "N_MEIOS", "ALVO_BIN"])
tratada = df[COLS_FINAL].copy()
ign_vazio_bruta = int(((disf["ignorados (código 9/99)"] + disf["vazios (NaN)"]).drop(ALVO)).sum())
antes_depois = pd.DataFrame({
    "bruta": [len(bruto), bruto.shape[1], dup_exatas, n_vazias, int(pd.to_numeric(bruto["IDADE_ANOS"]).max()), int(bruto_dias.max()), ign_vazio_bruta, int((bruto[ALVO].isna() | (bruto[ALVO] == "9")).sum())],
    "tratada": [len(tratada), tratada.shape[1], int(bruto.loc[~dup_mask.values].duplicated().sum()), int((tratada.isna().mean() == 1).sum()), int(tratada["IDADE_ANOS"].max()), int(tratada["DIAS_ATE_NOTIF"].max()),
                int(sum((tratada[c] == "Não informado").sum() for c in COLS_CAT if c != ALVO)), int(tratada["ALVO_BIN"].isna().sum())],
}, index=["linhas", "colunas", "linhas totalmente idênticas", "colunas 100% vazias", "idade máxima (anos)", "dias máx. ocorrência→notificação", "'Não informado' (ignorado + vazio, exceto alvo)", "linhas sem alvo (ignorado/vazio)"])
display(antes_depois)

os.makedirs("dados", exist_ok=True)
tratada.to_csv("dados/dados_tratados.csv", index=False, encoding="utf-8")
print("Base tratada salva em dados/dados_tratados.csv:", tratada.shape, "| a bruta continua intacta em", achado)

**Autora:** Ana Julia Prado

## 5. Análise exploratória (EDA)
Trabalhamos sobre a base **tratada** (`df`). Cada gráfico tem título, rótulos nos eixos e uma frase de leitura. A estatística mostra **associação, não causa**. Resultados valem para **notificações de 2024 de mulheres adultas com autor parceiro/ex**, não para todas as mulheres em situação de violência.

### 5.1 Visão geral

In [ ]:
print("Período:", df["DT_NOTIFIC"].min().date(), "a", df["DT_NOTIFIC"].max().date(), "| Registros:", len(df))
print("UFs presentes:", df["UF"].nunique(), "| Regiões:", df["REGIAO"].value_counts().to_dict())
valid = {"IDADE_ANOS": df.loc[~df["IDADE_IMPUTADA"], "IDADE_ANOS"], "DIAS_ATE_NOTIF": df["DIAS_ATE_NOTIF"].dropna(), "HORA": df["HORA"].dropna()}
display(pd.DataFrame(valid).describe().round(1))

### 5.2 Variáveis numéricas: tendência central, dispersão e forma
Média, mediana, moda, desvio-padrão, variância, coeficiente de variação, quartis, IQR, assimetria e curtose. Usamos só os valores originais (sem imputados nem inválidos).

In [ ]:
def resumo_num(s):
    s = s.dropna(); q1, q3 = s.quantile([.25, .75])
    return {"n": len(s), "média": s.mean(), "mediana": s.median(), "moda": s.mode().iloc[0], "desvio-padrão": s.std(), "variância": s.var(),
            "CV (%)": s.std() / s.mean() * 100 if s.mean() else np.nan, "mín": s.min(), "Q1": q1, "Q3": q3, "máx": s.max(), "IQR": q3 - q1,
            "assimetria": stats.skew(s), "curtose": stats.kurtosis(s)}
tab_num = pd.DataFrame({c: resumo_num(valid[c]) for c in COLS_NUM}).T.round(2)
display(tab_num)

fig, axes = plt.subplots(3, 2, figsize=(11, 8))
rotulos_x = {"IDADE_ANOS": "idade (anos)", "DIAS_ATE_NOTIF": "dias entre ocorrência e notificação (cortado em 60)", "HORA": "hora do dia (0 a 23)"}
for i, c in enumerate(COLS_NUM):
    v = valid[c] if c != "DIAS_ATE_NOTIF" else valid[c].clip(upper=60)
    axes[i, 0].hist(v, bins=range(int(v.min()), int(v.max()) + 2) if c != "IDADE_ANOS" else 30, color=COR)
    axes[i, 0].axvline(valid[c].mean(), color=COR2, ls="--", label="média"); axes[i, 0].axvline(valid[c].median(), color=COR3, label="mediana"); axes[i, 0].legend(fontsize=8)
    axes[i, 0].set_title(f"{c}: histograma"); axes[i, 0].set_xlabel(rotulos_x[c]); axes[i, 0].set_ylabel("notificações")
    axes[i, 1].boxplot(valid[c], vert=False, flierprops=dict(markersize=2)); axes[i, 1].set_title(f"{c}: boxplot (valores originais)"); axes[i, 1].set_xlabel(rotulos_x[c].replace(" (cortado em 60)", "")); axes[i, 1].set_yticks([])
plt.tight_layout(); plt.show()

 **Obs:** a **idade** é quase simétrica: média 35,2 e mediana 34 anos, com metade das mulheres entre 26 e 42 anos (IQR de 16) e assimetria de 0,90, uma cauda leve para as mais velhas. O **tempo até notificar** é o oposto: a mediana é **0 dia** (notifica-se no mesmo dia), mas a média é 19,6 dias por causa de uma minoria que demora meses ou anos (máximo de 1.814 dias, assimetria de 10,7 e coeficiente de variação de 527%). Aqui a média engana; a mediana descreve melhor. A **hora** só existe em 60% das notificações (63.475) e tem mediana às 15h, com mais casos no fim do dia (moda às 20h). **Consequências:** comparar grupos pela mediana de dias, não imputar dias nem hora, e usar `RobustScaler` ou log na modelagem. Obs.: como a mediana é 0 e o Q3 é 4, o IQR marca como outlier tudo acima de 10 dias (16.788 linhas, 16%): é um sinal de cauda longa, não de erro.

### 5.3 Variáveis categóricas: distribuição de frequências
Absoluta, relativa (%) e acumulada, com a moda de cada variável.

In [ ]:
def freq(s, ordem=None):
    f = s.value_counts(dropna=False)
    if ordem: f = f.reindex([o for o in ordem if o in f.index] + [i for i in f.index if i not in ordem])
    t = pd.DataFrame({"frequência": f, "%": (f / f.sum() * 100).round(1)}); t["% acumulado"] = t["%"].cumsum().round(1)
    return t
ORD_ESC = ["Analfabeta", "1ª-4ª série EF incompleta", "4ª série EF completa", "5ª-8ª série EF incompleta", "EF completo", "EM incompleto", "EM completo", "Superior incompleto", "Superior completo", "Não se aplica", "Não informado"]
for c, ordem in [("VINCULO", None), ("CS_RACA", None), ("CS_ESCOL_N", ORD_ESC), ("LOCAL_OCOR", None), ("REGIAO", None)]:
    print(f"\n== {c}  (moda: {df[c].mode().iloc[0]})"); display(freq(df[c], ordem))

tipos_tab = pd.DataFrame({"Sim": {v: int((df[k] == "Sim").sum()) for k, v in TIPOS_VIOL.items()}})
tipos_tab["% das notificações"] = (tipos_tab["Sim"] / len(df) * 100).round(1)
meios_tab = pd.DataFrame({"Sim": {v: int((df[k] == "Sim").sum()) for k, v in MEIOS_AGR.items()}})
meios_tab["% das notificações"] = (meios_tab["Sim"] / len(df) * 100).round(1)
print("\n== Tipos de violência (não excludentes: uma notificação pode ter vários)"); display(tipos_tab.sort_values("Sim", ascending=False))
print("== Meios de agressão (não excludentes)"); display(meios_tab.sort_values("Sim", ascending=False))

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (s, t, xl) in zip(axes.ravel(), [(df["VINCULO"], "Vínculo com o autor", "notificações"), (df["CS_RACA"], "Raça/cor", "notificações"),
                                         (tipos_tab["% das notificações"], "Tipos de violência (% das notificações)", "% das notificações"), (meios_tab["% das notificações"], "Meios de agressão (% das notificações)", "% das notificações")]):
    (s if pd.api.types.is_numeric_dtype(s) else s.value_counts()).sort_values().plot.barh(ax=ax, color=COR)
    ax.set_title(t); ax.set_xlabel(xl); ax.set_ylabel("")
plt.tight_layout(); plt.show()

 **Obs:** em **82,2%** das notificações o autor é o cônjuge (54,1%) ou o ex-cônjuge (28,1%); namorado(a) e ex somam 17,3%. A violência acontece na **residência** em 81,3% dos casos. Quanto ao tipo, a **física** aparece em 82,1% e a **psicológica/moral** em 46,2% (os tipos se somam: uma notificação pode ter vários); sexual 6,2%, financeira/econômica 5,6%. O meio mais comum é a **força corporal** (75,5%), seguido de **ameaça** (35,2%) e **enforcamento** (11,4%); arma de fogo aparece em 1,2%. Perfil: 46,3% são pardas e 36,8% brancas; o ensino médio completo é o nível mais frequente (28,3%), mas **27,7% não têm escolaridade informada**. O Sudeste concentra 52,6% das notificações; isso é contagem absoluta, **não** taxa por habitante, então não dá para dizer que lá há mais violência.

### 5.4 Série no tempo
Só há um ano (2024): olhamos mês, dia da semana e hora do dia. Não dá para falar de tendência de vários anos.

In [ ]:
serie = df.set_index("DT_NOTIFIC").resample("D").size()
mensal = df.groupby("MES_NOTIF").size()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(serie.index, serie.values, color=COR, lw=.6, alpha=.5, label="notificações/dia")
ax[0].plot(serie.index, serie.rolling(7, center=True).mean(), color=COR2, lw=2, label="média móvel de 7 dias")
ax[0].set_title("Notificações por dia em 2024"); ax[0].set_xlabel("data da notificação"); ax[0].set_ylabel("notificações"); ax[0].legend(fontsize=8)
ax[1].bar(mensal.index, mensal.values, color=COR); ax[1].axhline(mensal.mean(), color=COR2, ls="--", label="média mensal"); ax[1].legend(fontsize=8)
ax[1].set_xticks(range(1, 13)); ax[1].set_title("Total por mês"); ax[1].set_xlabel("mês de 2024"); ax[1].set_ylabel("notificações")
plt.tight_layout(); plt.show()

ordem_dia = ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]
sem = df["DIA_SEMANA"].value_counts().reindex(ordem_dia); horas = df["HORA"].dropna().astype(int).value_counts().sort_index()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].bar(sem.index, sem.values, color=COR); ax[0].set_title("Notificações por dia da semana"); ax[0].set_xlabel("dia da semana"); ax[0].set_ylabel("notificações")
ax[1].bar(horas.index, horas.values, color=COR3); ax[1].set_title(f"Hora da ocorrência (só os {int(df['HORA'].notna().sum())} registros com hora)"); ax[1].set_xlabel("hora do dia"); ax[1].set_ylabel("notificações")
plt.tight_layout(); plt.show()
print("Por mês:", mensal.to_dict()); print("Por dia da semana:", sem.to_dict())
print("Mês com mais:", mensal.idxmax(), mensal.max(), "| com menos:", mensal.idxmin(), mensal.min(), "| média mensal:", round(mensal.mean(), 1))
print("Entre 18h e 23h59:", f"{horas.loc[18:].sum() / horas.sum() * 100:.1f}%", "| entre 0h e 5h59:", f"{horas.loc[:5].sum() / horas.sum() * 100:.1f}%")

 **Obs:** em 2024 há pouca variação por mês: de **8.055** notificações em junho a **9.941** em janeiro, para uma média de 8.796 por mês. Por dia da semana, **segunda-feira** tem quase o dobro de registros do sábado (20.632 contra 10.689); como a data é a do preenchimento da ficha, uma hipótese (não testada aqui) é que parte dos episódios do fim de semana seja registrada na segunda. Entre os registros com hora, **39,3%** ocorreram entre 18h e 23h59 e 21,2% entre 0h e 5h59. Com um ano só, não dá para falar de tendência; um aumento ou queda pode refletir mais casos **ou** mais notificação.

### 5.5 Relações entre variáveis
Alvo: `OUT_VEZES` (1 = ocorreu outras vezes). Só entram as linhas com resposta Sim/Não.

In [ ]:
base = df.dropna(subset=["ALVO_BIN"]).copy()
print(f"Linhas com alvo conhecido: {len(base)} de {len(df)} ({len(base)/len(df)*100:.1f}%) | proporção de 'Sim': {base['ALVO_BIN'].mean()*100:.1f}%")

# (a) % de recorrência por grupo
def taxa(col, ordem=None):
    t = base.groupby(col)["ALVO_BIN"].agg(["mean", "size"]); t["mean"] = (t["mean"] * 100).round(1); t.columns = ["% ocorreu outras vezes", "n"]
    return t.reindex(ordem) if ordem else t.sort_values("% ocorreu outras vezes", ascending=False)
t_vinc = taxa("VINCULO"); t_idade = taxa("FAIXA_ETARIA", ["18-24", "25-34", "35-44", "45-59", "60+"]); t_reg = taxa("REGIAO")
for nome, t in [("vínculo", t_vinc), ("faixa etária", t_idade), ("região", t_reg)]: print(f"\n% de recorrência por {nome}:"); display(t)

# (b) fatores presentes × ausentes
fatores = {**TIPOS_VIOL, **MEIOS_AGR, "AUTOR_ALCO": "Autor com suspeita de álcool"}
linhas = []
for k, v in fatores.items():
    sim_, nao_ = base[base[k] == "Sim"]["ALVO_BIN"], base[base[k] == "Não"]["ALVO_BIN"]
    tab = pd.crosstab(base[k].where(base[k].isin(["Sim", "Não"])), base["ALVO_BIN"])
    chi2, p, _, _ = stats.chi2_contingency(tab); V = np.sqrt(chi2 / tab.values.sum())
    linhas.append((v, len(sim_), round(sim_.mean() * 100, 1), round(nao_.mean() * 100, 1), round((sim_.mean() - nao_.mean()) * 100, 1), round(V, 3), "<0,001" if p < .001 else round(p, 3)))
t_fat = pd.DataFrame(linhas, columns=["fator", "n com o fator", "% outras vezes (com)", "% outras vezes (sem)", "diferença (p.p.)", "V de Cramér", "p (qui-quadrado)"]).sort_values("diferença (p.p.)", ascending=False)
print("\nRecorrência com e sem cada fator (p.p. = pontos percentuais):"); display(t_fat)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
t_vinc["% ocorreu outras vezes"].sort_values().plot.barh(ax=ax[0], color=COR); ax[0].set_title("Recorrência por vínculo com o autor"); ax[0].set_xlabel("% que informou 'ocorreu outras vezes'"); ax[0].set_ylabel("")
t_idade["% ocorreu outras vezes"].plot.bar(ax=ax[1], color=COR3, rot=0); ax[1].set_title("Recorrência por faixa etária"); ax[1].set_xlabel("faixa etária (anos)"); ax[1].set_ylabel("% 'ocorreu outras vezes'")
t_fat.set_index("fator")["diferença (p.p.)"].sort_values().plot.barh(ax=ax[2], color=COR2); ax[2].set_title("Diferença na recorrência: com × sem o fator"); ax[2].set_xlabel("pontos percentuais"); ax[2].set_ylabel("")
plt.tight_layout(); plt.show()

# (c) numéricas por alvo e correlação de Spearman
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for a, c, xl in zip(ax, ["IDADE_ANOS", "DIAS_ATE_NOTIF"], ["idade (anos)", "dias até notificar"]):
    b = base.dropna(subset=[c]); a.boxplot([b.loc[b["ALVO_BIN"] == 0, c], b.loc[b["ALVO_BIN"] == 1, c]], showfliers=False)
    a.set_xticks([1, 2]); a.set_xticklabels(["Não", "Sim"]); a.set_xlabel("ocorreu outras vezes?"); a.set_ylabel(xl); a.set_title(f"{c} × recorrência")
plt.tight_layout(); plt.show()
cols_corr = ["IDADE_ANOS", "DIAS_ATE_NOTIF", "N_TIPOS", "N_MEIOS", "ALVO_BIN"]
corr = base[cols_corr].corr(method="spearman").round(2); display(corr)
fig, ax = plt.subplots(figsize=(5.4, 4.4)); im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_yticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right"); ax.set_yticklabels(corr.index); ax.grid(False)
for i in range(len(corr)):
    for j in range(len(corr)): ax.text(j, i, corr.iloc[i, j], ha="center", va="center", fontsize=8)
plt.colorbar(im, shrink=.8); plt.title("Correlação de Spearman (variáveis numéricas)"); plt.tight_layout(); plt.show()

**Obs:** entre as **89.988** notificações com resposta Sim/Não, **72,3%** dizem que a violência já tinha ocorrido antes. A recorrência é maior com **ex-cônjuge (76,5%)** e cônjuge (73,1%) e menor com namorado(a) (60,1%); cresce com a idade, de 66,8% (18-24 anos) para 79,4% (60+), e é maior no Nordeste (78,9%) que no Norte (68,7%). Entre os fatores, quem relata violência **financeira/econômica** (92,3% contra 71,0%), **ameaça** (84,2% contra 64,8%) e **psicológica/moral** (81,1% contra 63,9%) tem mais recorrência. Já a violência **física** vai no sentido contrário (70,1% contra 82,0%): uma possível explicação (hipótese, não testada) é que episódios físicos levam ao serviço de saúde já na primeira vez, enquanto a violência psicológica e as ameaças se repetem sem lesão que mande a mulher ao atendimento. Os tamanhos de efeito são pequenos (V de Cramér no máximo 0,21) e as correlações de Spearman com o alvo são fracas (0,07 a 0,13). **Atenção:** é associação em notificações de saúde; não mostra causa e não vale para todas as mulheres.

### 5.6 Três perguntas respondidas com os dados

In [ ]:
# Pergunta 1: quem é o autor e qual a proporção de recorrência?
p1 = df["VINCULO"].value_counts(normalize=True).mul(100).round(1)
rec_geral = base["ALVO_BIN"].mean() * 100
# Pergunta 2: ameaça, enforcamento e arma de fogo estão associados a mais recorrência?
p2 = t_fat.set_index("fator").loc[["Ameaça", "Enforcamento", "Arma de fogo", "Força corporal", "Psicológica/moral", "Sexual"], ["n com o fator", "% outras vezes (com)", "% outras vezes (sem)", "diferença (p.p.)"]]
# Pergunta 3: quanto tempo se leva para notificar, e quando se notifica mais?
dias = df["DIAS_ATE_NOTIF"].dropna()
p3 = {"mediana (dias)": dias.median(), "% no mesmo dia": (dias == 0).mean() * 100, "% em até 7 dias": (dias <= 7).mean() * 100, "% após 30 dias": (dias > 30).mean() * 100}
rec_dias = base.dropna(subset=["DIAS_ATE_NOTIF"]).assign(tarde=lambda d: d["DIAS_ATE_NOTIF"] > 30).groupby("ALVO_BIN")["tarde"].mean().mul(100).round(1)
print("P1 vínculo (%):", p1.head(5).to_dict(), "| recorrência geral (entre quem respondeu):", round(rec_geral, 1))
print("\nP2:"); display(p2)
print("P3:", {k: round(v, 1) for k, v in p3.items()}, "| % após 30 dias por recorrência (0 = Não, 1 = Sim):", rec_dias.to_dict())
print("   mês com mais notificações:", mensal.idxmax(), f"({mensal.max()}) vs. média mensal {mensal.mean():.0f}")

**Pergunta 1: quem é o autor mais frequente e com que frequência a violência se repete?** O cônjuge (54,1%) e o ex-cônjuge (28,1%) somam **82,2%** das notificações; namorado(a) 9,5% e ex-namorado(a) 7,8%. Entre as 89.988 que responderam, **72,3%** disseram que a violência já tinha ocorrido antes (ex-cônjuge 76,5%; namorado(a) 60,1%).

**Pergunta 2: ameaça, enforcamento e arma de fogo vêm com mais repetição?** **Ameaça**, sim: 84,2% de recorrência com ameaça contra 64,8% sem (+19,4 pontos percentuais; n = 34.304). **Enforcamento**, um pouco: 79,1% contra 71,3% (+7,8 p.p.). **Arma de fogo**, não há diferença clara: 73,5% contra 72,2% (+1,2 p.p., p = 0,38, n = 1.153). Nenhum desses números mostra que um meio *causa* repetição.

**Pergunta 3: quanto tempo se leva para notificar e quando se notifica mais?** A mediana é **0 dia**: **51,9%** são notificadas no mesmo dia, **81,4%** em até 7 dias e **9,3%** depois de 30 dias (entre as 104.931 com data de ocorrência válida). Nas notificações em que a violência se repetiu, 10,6% demoraram mais de 30 dias, contra 7,3% nas que não se repetiram. O mês com mais notificações foi **janeiro (9.941)**, contra média mensal de 8.796.

**Autora:** Ana Clara Mantella

## 6. Plano das Entregas 3 e 4: modelagem e avaliação
Quem assumiu **Modelagem e Avaliação** entrega aqui só o planejamento (nada é treinado). O plano respeita o que o diagnóstico mostrou.

In [ ]:
y = df["ALVO_BIN"].dropna()
baseline = max(y.mean(), 1 - y.mean())
print(f"Linhas com alvo conhecido: {len(y)} de {len(df)} | classe 'Sim': {y.mean():.1%} | classe 'Não': {1 - y.mean():.1%}")
print(f"Acurácia de um 'modelo' que SEMPRE chuta a classe majoritária ('Sim'): {baseline:.1%}  ← piso que qualquer modelo precisa superar")

plano = pd.DataFrame([
    ("Objetivo", "Estimar, a partir do registro de um episódio, se há padrão de repetição ('ocorreu outras vezes'), para o app **lembrar a usuária de completar o histórico no diário**. Não é um indicador de risco e não prevê violência futura"),
    ("Tipo de aprendizado", "Supervisionado, classificação binária (a base tem alvo registrado)"),
    ("Variável-alvo", f"`OUT_VEZES` → `ALVO_BIN` (1 = Sim, 0 = Não). {int(df['ALVO_BIN'].isna().sum())} linhas com 'Ignorado'/vazio ficam fora do treino"),
    ("Variáveis candidatas (entradas)", "Tipos de violência (`VIOL_*`), meios de agressão (`AG_*`), vínculo (`VINCULO`), suspeita de álcool do autor, local, número de envolvidos, faixa etária, `N_TIPOS`, `N_MEIOS`, `DIAS_ATE_NOTIF`"),
    ("NÃO entram", "`ENCAMINH` (REDE_SAU, DELEG_MULH, MPU...: acontecem **depois** do atendimento = vazamento); `DT_*` administrativas; `EVOLUCAO`/`CLASSI_FIN` (quase vazias); `raca_cor`, `escolaridade`, `REGIAO/UF` (risco de viés: decidir em grupo e com o professor); `OCORRENCIA_ANTIGA`"),
    ("Técnicas", "Regressão logística (base interpretável) → árvore/random forest → gradient boosting (`HistGradientBoostingClassifier`), comparando com o baseline"),
    ("Métricas", "**Recall da classe de interesse** e precisão, F1/F2, PR-AUC, matriz de confusão e calibração. Acurácia sozinha engana: o piso é " + f"{baseline:.1%}"),
    ("Validação", "Separar 20% para teste **antes** de imputar/escalonar; validação cruzada estratificada (5 partes) no treino; pré-processamento dentro de `Pipeline`; checagem extra: treinar em jan-set e testar em out-dez"),
    ("Desbalanceamento", f"Classe 'Sim' = {y.mean():.1%}: usar `class_weight='balanced'` e comparar com limiar ajustado; reamostrar **só no treino**"),
    ("Riscos", "Vazamento; viés de seleção (só quem chegou à saúde); 'Ignorado' não aleatório (14,5 mil linhas fora do alvo); modelo não é instrumento validado nem substitui o FONAR e profissionais; erro (falso negativo ou falso positivo) afeta pessoas"),
    ("Critério mínimo de utilidade", "Superar o baseline em recall e PR-AUC no teste, com resultado estável entre as partes da validação e sem depender de variável de vazamento"),
    ("Documentar sem exagero", "Escrever 'padrões associados à repetição em notificações de saúde'; nunca 'prevê risco', 'protege' ou 'evita violência'"),
    ("Decisão a confirmar com o professor", "O exemplo da disciplina exibe uma faixa 'baixa/média/alta atenção' no app. Para o BookSpace, sugerimos um aviso de completar histórico, sem rótulo de risco, e deixar a avaliação de risco a cargo do formulário baseado no FONAR (Entrega 4)"),
], columns=["item", "decisão"])
display(plano)

disp = pd.DataFrame([
    ("Tipos de violência (física, psicológica, sexual, patrimonial)", "Sim: categorias do diário", "Existe na base como VIOL_*"),
    ("Meios de agressão (ameaça, enforcamento, arma...)", "A definir: precisa de campo estruturado", "AG_*"),
    ("Vínculo com o autor", "A definir: pergunta no cadastro/registro", "VINCULO (REL_*)"),
    ("Suspeita de álcool do autor", "A definir", "AUTOR_ALCO"),
    ("Local do episódio", "A definir", "LOCAL_OCOR"),
    ("Dias entre o episódio e o registro", "Sim: calculável (data do fato × data do registro)", "DIAS_ATE_NOTIF"),
    ("Idade / faixa etária", "Opcional (cuidado LGPD)", "FAIXA_ETARIA"),
], columns=["variável", "o app consegue coletar?", "equivalente na base"])
print("\nCompatibilidade com o app (a validar com o time de desenvolvimento):"); display(disp)

**Autora:** Ana Julia Fernandes

## 7. Conclusões e checklist

In [ ]:
print("RESUMO DOS NÚMEROS:")
print(f"- Base bruta: {len(bruto)} linhas, {bruto.shape[1]} colunas; {len(vazias)} colunas 100% vazias; EVOLUCAO preenchida em {int(bruto['EVOLUCAO'].notna().sum())} linhas")
print(f"- Ignorados (códigos 9/99) nas colunas analisadas: {ign_total}; ocupação genérica: {ocup_99}; inválidos: idade {int(val['valores inválidos'].iloc[0])}, ocorrência > 5 anos {int(val['valores inválidos'].iloc[1])}")
print(f"- Duplicatas idênticas removidas: {n0 - len(df)} → base tratada com {len(df)} linhas")
print(f"- Alvo conhecido: {len(y)} linhas ({len(y)/len(df)*100:.1f}%); 'Sim' = {y.mean()*100:.1f}%; baseline = {baseline*100:.1f}%")

 **Conclusões:** a base bruta tinha **105.555 linhas e 160 colunas**; 14 estavam 100% vazias e `EVOLUCAO` e `CLASSI_FIN` (campos do que aconteceu depois) estavam preenchidas em só 2 e 12 linhas. Ou seja, **a base não permite estudar desfecho posterior**, o que confirma que o objetivo "prever violência grave futura" não cabe aqui. No diagnóstico: 191.313 códigos "ignorado" (9/99) nas colunas analisadas (os maiores: escolaridade 23,8%, álcool do autor 20,9%, motivação 19,2% e o próprio alvo 13,8%), 52,5% das ocupações preenchidas com código genérico, 4 idades acima de 100 anos, 621 ocorrências com mais de 5 anos antes da notificação (a mais antiga de 1968), 1.961 casos em que idade e ano de nascimento não batem, 1.420 com subtipo sexual preenchido sem violência sexual e 3 linhas idênticas (removidas). Depois do tratamento ficaram **105.552 linhas e 62 colunas**, sem coluna vazia, com município, ano de nascimento e textos livres fora do arquivo. **Na EDA:** vínculo íntimo em 82,2% (cônjuge/ex-cônjuge), residência em 81,3%, recorrência de **72,3%** entre as 89.988 que responderam, mais alta com ameaça (+19,4 p.p.) e violência psicológica (+17,1 p.p.). **Limitações:** subnotificação e viés de seleção (só quem chegou a um serviço que notifica), 14,7% sem resposta no alvo, comparação do total com o TabNet ainda pendente, um ano só e associação sem causa. **Próximo passo:** na Entrega 3, treinar o modelo de recorrência e comparar com o baseline de 72,3%.

**Checklist da Entrega 2:** 

- [x] Base escolhida e justificada (SINAN está entre as sugeridas: DataSUS); mas confirmar com o professor o uso do **microdado** em vez do painel
- [ ] Ficha da base preenchida (**link do download e licença falta preencher**), aviso acadêmico no README
- [x] Dicionário de dados completo (160 colunas) com tipo estatístico
- [x] Diagnóstico nas 5 dimensões, com contagens e exemplos (acurácia: preencher `TOTAL_TABNET`)
- [x] Tratamento documentado (decisões e antes × depois); base bruta preservada e tratada em `data-science/dados/`
- [x] EDA: medidas, frequências, série no tempo e relações, cada gráfico com título, eixos e frase de leitura
- [x] Três perguntas respondidas com números, sem concluir causa a partir de correlação
- [ ] Cada seção assinada por quem a fez; commits individuais de todos os integrantes de Data Science
- [x] Notebook executa do zero (*Restart and run all*) sem erro; nenhum dado pessoal nem chave no repositório
- [x] `USO_IA.md` e Kanban atualizados
